This model uses a Convolutional Neural Network design in order to classify images, using various convolutions that are developed and refined overtime across several training sessions in order to properly classify images into various categories.

The programming will be done in two ways. First, pytorch will be used in order to build a simple CNN. Then, it'll be done from scratch using numpy as the mathematics library.

Training data will be stored in a folder named "Datasets" and in two folders - "Cat" and "Dog". The dataset was obtained from here: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset?resource=download.

Test data will be stored within "Datasets" in "\_TEST\_", with the same format as is used for the training data. The dataset was obtained from here:
https://www.kaggle.com/datasets/abdullahbakr7/cats-vs-dogs-image-classification/data

In [2]:
# Initial Imports.
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import numpy as np
import os

In [40]:
# Setting torch to use GPU acceleration if possible.
device = torch.device("cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")

torch.set_default_device(device)
print(f"Using device: {torch.get_default_device()}")

Using device: cpu


**Pytorch Model**

In [41]:
# Preparing dataset for Pytorch
class PytorchDataset(Dataset):
    def __init__(self, root = "Datasets"):
        self.transform = transforms.Compose({
            transforms.Resize((32, 32)),
            transforms.ToTensor()
        })

        self.images = []
        self.labels = []

        self.classes = {
            "Cat": 0,
            "Dog": 1
        }

        # Load dataset
        for className, label in self.classes.items():
            classPath = os.path.join(root, className)

            for imageName in os.listdir(classPath):
                imagePath = os.path.join(classPath, imageName)
                self.images.append(imagePath)
                self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        imagePath = self.images[idx]

        # Load and prepare image.
        image = Image.open(imagePath).convert("RGB")
        image = self.transform(image)

        label = self.labels[idx]

        return image, label

In [42]:
# Generate the dataset. This dataset can then be used in as many models as we want.
dataset = PytorchDataset()

In [43]:
class PytorchModel(nn.Module):
    def __init__(self):
        super(PytorchModel, self).__init__()
        self.dataset = dataset
        
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.l1 = nn.Linear(16 * 5 * 5, 120)
        self.l2 = nn.Linear(120, 84)
        self.l3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        return self.l3(x)

In [49]:
# Prepare Training.
model = PytorchModel()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = 0.9)

trainLoader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [50]:
# Train the model.
epochs = 50

for epoch in range(epochs):
    runningLoss = 0.0

    for i, data in enumerate(trainLoader):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        runningLoss += loss.item()
        
    print("Epoch #" + str(epoch + 1) + ": " + str(runningLoss / (i + 1)))

Epoch #1: 0.8569628706063761
Epoch #2: 0.6972201862146178
Epoch #3: 0.6916871057142078
Epoch #4: 0.6879117887495728
Epoch #5: 0.6834235925351262
Epoch #6: 0.676551914428506
Epoch #7: 0.6719291611858036
Epoch #8: 0.6694031382155845
Epoch #9: 0.6654498059578868
Epoch #10: 0.6616954214280218
Epoch #11: 0.6584751423057693
Epoch #12: 0.6528989552232005
Epoch #13: 0.6486266833132185
Epoch #14: 0.6452107779738848
Epoch #15: 0.6386637788294526
Epoch #16: 0.6351612894736287
Epoch #17: 0.6291092509961189
Epoch #18: 0.6253044179181004
Epoch #19: 0.6178557050731176
Epoch #20: 0.6121788419344846
Epoch #21: 0.6074909523243794
Epoch #22: 0.5956708235890055
Epoch #23: 0.587616338944801
Epoch #24: 0.579742373693782
Epoch #25: 0.5726036923696928
Epoch #26: 0.560346063140713
Epoch #27: 0.5535015456206963
Epoch #28: 0.5479954588596169
Epoch #29: 0.5394586218744898
Epoch #30: 0.5330769811825984
Epoch #31: 0.5287493561844692
Epoch #32: 0.5204749433966853
Epoch #33: 0.5128717956030765
Epoch #34: 0.5068889599

In [46]:
# Test the model.
testDataset = PytorchDataset(os.path.join("Datasets", "_TEST_"))
testLoader = DataLoader(testDataset, batch_size = 4, shuffle = False)

correct = 0
total = 0

with torch.no_grad():
    for data in testLoader:
        images, labels = data
        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy: " + str(100 * correct / total))

Accuracy: 59.8


**Numpy Model**

In [416]:
# Initial imports.
import math

In [417]:
# Defining layers that we'll use throughout the mode. This includes Conv2D, Pooling, ReLu, and Linear.
class Layer:
    def __init__(self):
        self.parameters = {}

    def setup(self):
        self.gradients = {}
        for parameter in self.parameters.keys():
            self.gradients[parameter] = None

    def forward(self, x):
        return x

    def backward(self, lossGradient):
        return lossGradient

    def adjust_parameters(self, learningRate):
        for parameter in self.parameters.keys():
            self.parameters[parameter] -= self.gradients[parameter] * learningRate

In [418]:
class Linear(Layer):
    def __init__(self, inputSize, outputSize):
        self.parameters = {
            "Weights": np.random.rand(inputSize, outputSize),
            "Bias": np.random.rand(1, outputSize)
        }

        self.prevInput = None
        
        self.setup()

    def forward(self, x):
        self.prevInput = x
        return np.matmul(x, self.parameters["Weights"]) + self.parameters["Bias"]

    def backward(self, lossGradient):
        self.gradients["Weights"] = np.repeat(np.transpose(self.prevInput), self.parameters["Weights"].shape[1], axis = 1)
        self.gradients["Weights"] *= np.repeat(lossGradient, self.parameters["Weights"].shape[0], axis = 0)
        self.gradients["Bias"] = lossGradient

        return np.matmul(lossGradient, np.transpose(self.parameters["Weights"]))

In [580]:
class Conv2D(Layer):
    def __init__(self, kernel, stride = 1):
        self.kernel = kernel
        self.stride = stride

        self.parameters = {
            "Filter": np.random.rand(*kernel)
        }

        self.prevInput = None
        self.setup()

    def forward(self, x):
        self.prevInput = x
        output = []

        row = 0
        col = 0

        while row <= x.shape[0] - self.kernel[0]:
            newRow = []

            while col <= x.shape[1] - self.kernel[1]:
                targetInput = x[row:(row + self.kernel[0]), col:(col + self.kernel[1])]
                value = np.sum(targetInput * self.parameters["Filter"])
                newRow.append(value)

                col += self.stride

            row += self.stride
            col = 0
            output.append(newRow)

        return np.array(output)

    def backward(self, lossGradient):
        rows = lossGradient.shape[0]
        cols = lossGradient.shape[1]

        gradient = np.zeros_like(self.parameters["Filter"])
        output = np.zeros_like(self.prevInput)

        cRow = 0
        cCol = 0

        for row in range(rows):
            for col in range(cols):
                subMatrix = self.prevInput[cRow:(cRow + self.kernel[0]), cCol:(cCol + self.kernel[1])]
                gradient += subMatrix * lossGradient[row, col]
                
                output[cRow:(cRow + self.kernel[0]), cCol:(cCol + self.kernel[1])] += self.parameters["Filter"] * lossGradient[row, col]

                cCol += self.stride

            cRow += self.stride
            cCol = 0

        self.gradients["Filter"] = gradient
        return output

In [581]:
class ReLU(Layer):
    def __init__(self):
        self.prevInput = None
        self.parameters = {}
        self.setup()

    def forward(self, x):
        self.prevInput = x
        return np.maximum(0, x)

    def backward(self, lossGradient):
        output = np.zeros_like(self.prevInput)
        output[self.prevInput > 0] = 1
        return output * lossGradient

In [582]:
class MaxPool(Layer):
    def __init__(self, kernel, stride = 1):
        self.kernel = kernel
        self.stride = stride
        
        self.prevInput = None
        self.parameters = {}
        self.setup()

    def forward(self, x):
        self.prevInput = x
        output = []

        row = 0
        col = 0

        while row <= x.shape[0] - self.kernel[0]:
            newRow = []

            while col <= x.shape[1] - self.kernel[1]:
                value = np.max(x[row:(row + self.kernel[0]), col:(col + self.kernel[1])])
                newRow.append(value)

                col += self.stride

            row += self.stride
            col = 0
            output.append(newRow)

        return np.array(output)

    def backward(self, lossGradient):
        rows = lossGradient.shape[0]
        cols = lossGradient.shape[1]

        output = np.zeros_like(self.prevInput)

        cRow = 0
        cCol = 0

        for row in range(rows):
            for col in range(cols):
                subMatrix = self.prevInput[cRow:(cRow + self.kernel[0]), cCol:(cCol + self.kernel[1])]
                maxIndex = np.unravel_index(np.argmax(subMatrix), subMatrix.shape)
                output[cRow + maxIndex[0], cCol + maxIndex[1]] += lossGradient[row, col]

                cCol += self.stride

            cRow += self.stride
            cCol = 0

        return output

In [611]:
# Construct our dataset, which will be responsible for holding and converting images.
class Dataset:
    def __init__(self, imageDimensions = [32, 32]):
        self.imageDimensions = imageDimensions
        self.load_dataset()

    def load_dataset(self):
        # Should only be run once, as it takes a while.
        self.dataset = {
            "Cat": [],
            "Dog": []
        }

        for folder in self.dataset.keys():
            targetFolder = os.path.join("Datasets", folder)
            for file in os.listdir(targetFolder):
                imagePath = os.path.join(targetFolder, file)
                self.dataset[folder].append(self.convert_image(imagePath))

    def convert_image(self, imagePath):
        image = Image.open(imagePath)
        image = image.resize((self.imageDimensions[0], self.imageDimensions[1]))
        return np.array(image)

In [612]:
dataset = Dataset()

C:\Users\conta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [ ]:
class Model:
    def __init__(self):
        pass

    def forward(self, x):
        pass

    def backward(self, loss):
        pass

    def adjust_parameters(self, learningRate):
        pass